## Notebook for generate batches of image from the CosmoDC2 catalog. 

### In this notebook, we will focus on a larger scale production, instead of for demostration purpose

In [ ]:
import btk
import btk.survey
import btk.draw_blends
import btk.catalog
import btk.sampling_functions
import cv2
import numpy as np
from galcheat.utilities import mean_sky_level
import matplotlib.pyplot as plt

In [ ]:
from astropy.io import fits
from astropy.table import Table, vstack
import h5py
import json


In [ ]:
config = {}

In [ ]:
"size" in config.keys()

In [ ]:



def catalog_to_batch(catalog, config, seed = 0):
    
    
    
    stamp_size = 24.0 if "stamp_size" not in config.keys() else config['stamp_size']
    max_number = 100 if "max_number" not in config.keys() else config['max_number']
    min_mag = 20.0 if "min_mag" not in config.keys() else config['min_mag']
    max_mag = 25.3 if "max_mag" not in config.keys() else config['max_mag']
    batch_size = 100 if "batch_size" not in config.keys() else config['batch_size']
    survey_name = "LSST" if "survey_name" not in config.keys() else config['survey_name']
    
    sampling_function = btk.sampling_functions.RandomSquareSampling(
            max_number=max_number,  # always get `max_number` galaxies
            stamp_size=stamp_size,
            min_mag = min_mag, max_mag = max_mag,
            seed = seed)
    
    survey = btk.survey.get_surveys("LSST")
    
    draw_generator = btk.draw_blends.CatsimGenerator(
        catalog,
        sampling_function,
        survey,
        batch_size=batch_size,
        stamp_size=stamp_size,
        njobs=1,
        add_noise="all",
        seed=seed, # use same seed here
    )

    # generate batch 100 blend catalogs and images.
    blend_batch = next(draw_generator)
    
    
    return blend_batch
    
    

In [ ]:
def get_bbox(mask):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    return rmin-1, rmax+1, cmin-1, cmax+1


def create_metadata(blend_batch, sky_level,filename, sn, idx, batch_id):

    """ Code to format the metadatain to a dict.  It takes the i-band and makes a footprint+bounding boxes
    from thresholding to sn*sky_level
    
    Parameters
    
    blend_batch: BTK blend batch
        BTK batch of blends
    sky_level: float
        The background sky level in the i-band
    sn: int
        The signal-to-noise ratio for thresholding
    idx:
        The index of the blend in the blend_batch
        
    Returns
        ddict: dict
            The dictionary of metadata for the idx'th blend in the batch 
    
    """
    

    ddict = {}

    ddict[f"file_name"] = filename
    ddict["image_id"] = idx+batch_id*config['batch_size']
    ddict["height"] = blend_batch.isolated_images.shape[-2]
    ddict["width"] = blend_batch.isolated_images.shape[-1]

    iso_images = blend_batch.isolated_images[:,:, 3] # only i-band for now
    segs = btk.metrics.utils.get_segmentation(iso_images, sky_level, sigma_noise=sn)
#     print(segs.shape)
#     plt.imshow(segs[0][1])


    cat = blend_batch.catalog_list[idx]
    n = len(cat)
    objs = []
    for j in range(n):
        detect = iso_images[idx][j]
        mask = segs[idx][j]

        bbox = get_bbox(mask)
        mask = segs[idx][j]
        x0 = bbox[2]
        x1 = bbox[3]
        y0 = bbox[0]
        y1 = bbox[1]


        w = x1-x0
        h = y1-y0
        bbox = [x0, y0, w, h]

        redshift = cat[j]['redshift']

        contours, hierarchy = cv2.findContours(
                    (mask).astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
                )


        segmentation = []
        for contour in contours:
            # contour = [x1, y1, ..., xn, yn]
            contour = contour.flatten()
            if len(contour) > 4:
                segmentation.append(contour.tolist())
        # No valid countors
        if len(segmentation) == 0:
            continue

        obj = {
            "bbox": bbox,
            "area": w*h,
            #"bbox_mode": BoxMode.XYWH_ABS,
            "bbox_mode": 1,
            "segmentation": segmentation,
            "category_id": 0,
            "redshift": redshift
        }
        objs.append(obj)

    ddict['annotations'] = objs

    return ddict

In [ ]:
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

In [ ]:
catalog_name = "data/btk_catalog_1deg_6bands.fits" # contains ~85k entries

catalog = btk.catalog.CatsimCatalog.from_file(catalog_name)


In [ ]:
# produce the image in batches, write the flatten image to the flatten_array, and 
# append the metadata to metadata list. del blend_batch after each batch to save
# memory. 

number_of_batches = 50

metadata = []
hdf5_filename = f"./data/dc2/dc2_100images.hdf5"
metadata_filename = "data/dc2/dc2_100images_meta.json"
config = {"batch_size":2, "stamp_size": 60}

metadata = []
flatten_array = np.zeros((number_of_batches*config['batch_size'], int(round(6*config['stamp_size']**2/0.2**2))))


In [ ]:

for batch_id in range( number_of_batches):
    print(batch_id)

    blend_batch = catalog_to_batch(catalog, config, seed = batch_id)
    sky_level = mean_sky_level(blend_batch.survey, "i").to_value('electron')
    
    flatten_array[batch_id*config['batch_size']:(batch_id+1)*config['batch_size']] = blend_batch.blend_images.reshape(blend_batch.blend_images.shape[0], -1)
    
    for idx in range(config['batch_size']):
        ddict = create_metadata(blend_batch,sky_level,hdf5_filename,sn=2,idx=idx, batch_id = batch_id)
        metadata.append(ddict)
        
    del blend_batch

In [ ]:
# Save the batch as a hdf5

h5f = h5py.File(hdf5_filename, 'w')
h5f.create_dataset('image', data=flatten_array)

h5f.close()

In [ ]:
# Save the metadata

with open(metadata_filename, "w") as f:
    json.dump(metadata, f, cls = NpEncoder)